# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohsanalee/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

### 1. Unit of analysis + time window

* **One row means:** one daily observation for one content item for one client on one `report_date`.
* **Table used:** `fact_content_daily_performance`, joined to `dim_content` or `dim_clients` only when client/content context is needed.
* **Decision window:** use the previous 30 days of performance as the feature window.
* **Outcome window:** use the following 30 days as the future outcome window.
* **Prediction target:** whether the content item's impressions decline by more than 20% in the outcome window compared with the previous 30-day window.
* **Deliberately excluded:** `trend_direction` and `trend_pct`, because they are derived from the outcome/trend information and would leak the label into the features.


In [39]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
%pip -q install duckdb huggingface_hub pandas scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Fields: feature / label / context / excluded

### 2. Fields: feature / label / context / excluded

**Features — safe before the decision moment**

* `imp_prev30`: impressions observed during the previous 30 days.
* `clk_prev30`: clicks observed during the previous 30 days.
* `pos_prev30`: average search position observed during the previous 30 days.
* `visible_queries`: number of visible queries associated with the content.
* `top_query_share`: share of impressions coming from the highest-impression query.

**Label / proxy**

* `is_declining`: 1 when outcome-window impressions are less than 80% of previous-30-day impressions; otherwise 0.

**Context**

* `client_hash_id`: identifies the client for grouping, joining, and client-level validation.
* `content_hash_id`: identifies the content item for grouping and joining.
* `report_date`: identifies when the observation occurred.

**Excluded**

* `trend_direction` and `trend_pct`: excluded because they contain information derived from the performance trend/label.
* `is_declining`: excluded from the feature set because it is the target itself.
* Future outcome-window performance: excluded because it is not knowable at the decision moment.


In [40]:
import os
import getpass
import duckdb
import pandas as pd

# Get Hugging Face token from environment variable.
HF_TOKEN = os.environ.get("HF_TOKEN")

# If HF_TOKEN is not available, ask for it privately.
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass(
        "Enter your Hugging Face READ token (hf_...): "
    )

# Create DuckDB connection.
con = duckdb.connect()

# Give DuckDB access to the gated Hugging Face dataset.
con.execute(
    f"""
    CREATE OR REPLACE SECRET hf
    (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    )
    """
)

# Dataset location.
REL = "hf://datasets/FlyRank/internship-warehouse"

print("DuckDB connected to FlyRank warehouse.")

DuckDB connected to FlyRank warehouse.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [41]:
import getpass

HF_TOKEN = getpass.getpass("Enter your Hugging Face READ token: ")

HF_TOKEN = HF_TOKEN.strip()

if HF_TOKEN.startswith("Bearer "):
    HF_TOKEN = HF_TOKEN[len("Bearer "):].strip()

print("Token loaded:", bool(HF_TOKEN))
print("Correct HF token format:", HF_TOKEN.startswith("hf_"))

Token loaded: True
Correct HF token format: True


In [42]:
from huggingface_hub import snapshot_download

path = snapshot_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    token=HF_TOKEN,
    allow_patterns=[
        "fact_content_daily_performance/month=2026-03/*"
    ],
    force_download=True,
)

print("March 2026 daily files downloaded.")
print(path)

Fetching 1 files: 100%|██████████| 1/1 [01:03<00:00, 63.50s/it]

March 2026 daily files downloaded.
C:\Users\meerm\.cache\huggingface\hub\datasets--FlyRank--internship-warehouse\snapshots\50cbf7c3909d07be4d1b5906b4d09e882e5acbf2


In [43]:
from pathlib import Path

root = Path(path)

march_files = list(
    root.rglob("fact_content_daily_performance/month=2026-03/*.parquet")
)

print("March files found:", len(march_files))

for f in march_files:
    print(f)

March files found: 1
C:\Users\meerm\.cache\huggingface\hub\datasets--FlyRank--internship-warehouse\snapshots\50cbf7c3909d07be4d1b5906b4d09e882e5acbf2\fact_content_daily_performance\month=2026-03\data_0.parquet


In [44]:
TABLES["fact_daily"] = march_files[0]

print("March daily-performance file connected.")

March daily-performance file connected.


In [45]:
march_check = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM read_parquet('{TABLES["fact_daily"]}')
""").df()

march_check

,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


In [46]:
grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM read_parquet('{march_files[0]}')
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

grain_check

,report_date,client_hash_id,content_hash_id,row_count


In [47]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS available_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS NOT TRUE
        ) AS unavailable_rows
    FROM read_parquet('{march_files[0]}')
""").df()

availability_check

,total_rows,available_rows,unavailable_rows
0,9841378,413966,9427412


In [48]:
from huggingface_hub import snapshot_download

path_prev = snapshot_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    token=HF_TOKEN,
    allow_patterns=[
        "fact_content_daily_performance/month=2026-02/*"
    ],
    force_download=True,
)

print("February 2026 files downloaded.")
print(path_prev)

Fetching 1 files: 100%|██████████| 1/1 [00:45<00:00, 45.04s/it]

February 2026 files downloaded.
C:\Users\meerm\.cache\huggingface\hub\datasets--FlyRank--internship-warehouse\snapshots\50cbf7c3909d07be4d1b5906b4d09e882e5acbf2


In [49]:
from pathlib import Path

prev_root = Path(path_prev)

feb_files = list(
    prev_root.rglob(
        "fact_content_daily_performance/month=2026-02/*.parquet"
    )
)

print("February files found:", len(feb_files))

for f in feb_files:
    print(f)

February files found: 1
C:\Users\meerm\.cache\huggingface\hub\datasets--FlyRank--internship-warehouse\snapshots\50cbf7c3909d07be4d1b5906b4d09e882e5acbf2\fact_content_daily_performance\month=2026-02\data_0.parquet


In [50]:
from huggingface_hub import snapshot_download

path_next = snapshot_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    token=HF_TOKEN,
    allow_patterns=[
        "fact_content_daily_performance/month=2026-04/*"
    ],
    force_download=True,
)

print("April 2026 files downloaded.")
print(path_next)

Fetching 1 files: 100%|██████████| 1/1 [01:02<00:00, 62.50s/it]

April 2026 files downloaded.
C:\Users\meerm\.cache\huggingface\hub\datasets--FlyRank--internship-warehouse\snapshots\50cbf7c3909d07be4d1b5906b4d09e882e5acbf2


In [51]:
from pathlib import Path

next_root = Path(path_next)

apr_files = list(
    next_root.rglob(
        "fact_content_daily_performance/month=2026-04/*.parquet"
    )
)

print("April files found:", len(apr_files))

for f in apr_files:
    print(f)

April files found: 1
C:\Users\meerm\.cache\huggingface\hub\datasets--FlyRank--internship-warehouse\snapshots\50cbf7c3909d07be4d1b5906b4d09e882e5acbf2\fact_content_daily_performance\month=2026-04\data_0.parquet


In [52]:
features = con.sql(f"""
    WITH daily AS (
        SELECT *
        FROM read_parquet([
            '{feb_files[0]}',
            '{march_files[0]}'
        ])
    ),

    prev30 AS (
        SELECT
            client_hash_id,
            content_hash_id,

            SUM(
                COALESCE(gsc_impressions, 0)
            ) AS imp_prev30,

            SUM(
                COALESCE(gsc_clicks, 0)
            ) AS clicks_prev30,

            AVG(gsc_avg_position) AS avg_position_prev30

        FROM daily
        WHERE report_date >= DATE '2026-03-02'
          AND report_date <= DATE '2026-03-31'
        GROUP BY 1, 2
    )

    SELECT *
    FROM prev30
    WHERE imp_prev30 > 0
""").df()

features.head()

,client_hash_id,content_hash_id,imp_prev30,clicks_prev30,avg_position_prev30
0,client_73cda7b4e4f265ea,content_2cca146d1e3992e3,616.0,2.0,7.143161
1,client_73cda7b4e4f265ea,content_8f24b1a3279ad5c7,383.0,1.0,37.378907
2,client_73cda7b4e4f265ea,content_1736b39baa712103,475.0,0.0,23.888963
3,client_73cda7b4e4f265ea,content_dfa3bc2112719050,346.0,0.0,10.998035
4,client_73cda7b4e4f265ea,content_69716b92afaf7631,1443.0,0.0,6.707141


In [53]:
query_path = Path(path) / "fact_content_query_90d.parquet"

query_features = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        MAX(content_visible_query_count) AS visible_queries,
        MAX(anonymized_impressions_share) AS top_query_share
    FROM read_parquet('{query_path}')
    GROUP BY 1, 2
""").df()

feature_frame = features.merge(
    query_features,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

feature_cols = [
    "imp_prev30",
    "clicks_prev30",
    "avg_position_prev30",
    "visible_queries",
    "top_query_share",
]

feature_frame = feature_frame[
    ["client_hash_id", "content_hash_id"] + feature_cols
]

feature_frame.head()

,client_hash_id,content_hash_id,imp_prev30,clicks_prev30,avg_position_prev30,visible_queries,top_query_share
0,client_73cda7b4e4f265ea,content_2cca146d1e3992e3,616.0,2.0,7.143161,5.0,0.891353
1,client_73cda7b4e4f265ea,content_8f24b1a3279ad5c7,383.0,1.0,37.378907,8.0,0.605195
2,client_73cda7b4e4f265ea,content_1736b39baa712103,475.0,0.0,23.888963,21.0,0.350742
3,client_73cda7b4e4f265ea,content_dfa3bc2112719050,346.0,0.0,10.998035,4.0,0.703457
4,client_73cda7b4e4f265ea,content_69716b92afaf7631,1443.0,0.0,6.707141,22.0,0.861520


In [54]:
feature_frame.shape

(176268, 7)

In [55]:
feature_frame[feature_cols].isna().sum()


imp_prev30                 0
clicks_prev30              0
avg_position_prev30        0
visible_queries        73733
top_query_share        73733
dtype: int64

In [56]:
TABLES = {
    "dim_clients": f"read_parquet('{path}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{path}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{path}/fact_content_daily_performance/month=2026-03/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{path}/fact_content_query_90d.parquet')",
}

print("Local warehouse tables ready.")

Local warehouse tables ready.


In [57]:
# Build the April outcome for each client-content pair.
outcomes = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(COALESCE(gsc_impressions, 0)) AS imp_next30
    FROM read_parquet('{apr_files[0]}')
    WHERE report_date >= DATE '2026-04-01'
      AND report_date <= DATE '2026-04-30'
    GROUP BY 1, 2
""").df()

# Merge the future outcome with the March feature frame.
model_data = feature_frame.merge(
    outcomes,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

# Label: impressions decline by more than 20% in the next 30 days.
model_data["label"] = (
    model_data["imp_next30"]
    < 0.80 * model_data["imp_prev30"]
).astype(int)

# Fill missing query-level features with 0.
model_data["visible_queries"] = model_data["visible_queries"].fillna(0)
model_data["top_query_share"] = model_data["top_query_share"].fillna(0)

print("Model rows:", len(model_data))
print("Positive labels:", model_data["label"].sum())
print("Negative labels:", (model_data["label"] == 0).sum())

Model rows: 176268
Positive labels: 91174
Negative labels: 85094


In [58]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

feature_cols = [
    "imp_prev30",
    "clicks_prev30",
    "avg_position_prev30",
    "visible_queries",
    "top_query_share",
]

X = model_data[feature_cols]
y = model_data["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

honest_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

honest_model.fit(X_train, y_train)

honest_pred = honest_model.predict_proba(X_test)[:, 1]

honest_auc = roc_auc_score(y_test, honest_pred)

print(f"Honest ROC-AUC: {honest_auc:.3f}")

Honest ROC-AUC: 0.755


In [59]:
leak_data = model_data.copy()

# DELIBERATE LEAKAGE — this column directly comes from the label.
leak_data["leak_label"] = leak_data["label"]

leak_features = feature_cols + ["leak_label"]

X_leak = leak_data[leak_features]
y_leak = leak_data["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X_leak,
    y_leak,
    test_size=0.25,
    random_state=42,
    stratify=y_leak
)

leak_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

leak_model.fit(X_train, y_train)

leak_pred = leak_model.predict_proba(X_test)[:, 1]

leak_auc = roc_auc_score(y_test, leak_pred)

print(f"Leaked ROC-AUC: {leak_auc:.3f}")

Leaked ROC-AUC: 1.000


In [60]:
# Remove the deliberately leaked column.
leak_data = leak_data.drop(columns=["leak_label"])

print("Leak removed.")
print("Final model features:")
print(feature_cols)
print(f"Honest ROC-AUC retained: {honest_auc:.3f}")

Leak removed.
Final model features:
['imp_prev30', 'clicks_prev30', 'avg_position_prev30', 'visible_queries', 'top_query_share']
Honest ROC-AUC retained: 0.755


## 4. Data limits

### Data limitation

One important limitation is that the panel is unbalanced: not every client-content pair has the same amount of historical data. Some rows also have limited Google Search Console availability, as shown by the `ga4_data_available IS TRUE` check. Therefore, missing history or unavailable measurements can make comparisons less reliable, and the model should not be interpreted as having complete information for every content item.

In [61]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.